# 📈 Mini-Project 1: Sales Data Analysis

**Objective**: Put your Pandas and Data Cleaning skills to the test by analyzing a messy sales dataset.

**Skills Tested**:
- Pandas DataFrame manipulation
- Handling missing values
- Date parsing
- Basic aggregations
- Matplotlib visualizations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")

## 1. Generate Synthetic Messy Data
Run the cell below to generate the dataset we will clean.

In [ ]:
np.random.seed(42)

dates = pd.date_range(start='2023-01-01', end='2023-12-31', freq='D')
n_rows = len(dates)

messy_data = pd.DataFrame({
    'Date': dates.astype(str), # Make them strings to simulate bad parsing
    'ProductID': np.random.choice(['A100', 'B200', 'C300', 'D400'], n_rows),
    'Quantity': np.random.randint(1, 20, n_rows).astype(float),
    'Price': np.random.uniform(10.0, 500.0, n_rows)
})

# Introduce messy data
messy_data.loc[10:15, 'Quantity'] = np.nan
messy_data.loc[50:55, 'Price'] = np.nan
messy_data.loc[100, 'Price'] = 999999.0 # Outlier
messy_data.loc[200, 'Quantity'] = -5.0  # Invalid

print("Dataset shape:", messy_data.shape)
messy_data.head()

## 2. Data Cleaning
Your turn! Clean the dataset by handling the missing values and outliers.

In [ ]:
# TODO: Convert 'Date' column back to datetime objects
df_clean = messy_data.copy()
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# TODO: Handle negative quantities (replace with absolute value or drop)
df_clean['Quantity'] = df_clean['Quantity'].abs()

# TODO: Fill missing quantities with the median quantity
df_clean['Quantity'] = df_clean['Quantity'].fillna(df_clean['Quantity'].median())

# TODO: Fill missing prices with the mean price of that specific ProductID
df_clean['Price'] = df_clean.groupby('ProductID')['Price'].transform(lambda x: x.fillna(x.mean()))

# TODO: Remove extreme price outliers (e.g. Price > 1000)
df_clean = df_clean[df_clean['Price'] < 1000]

print("Missing values after cleaning:", df_clean.isnull().sum().sum())

## 3. Feature Engineering
Create new features for analysis.

In [ ]:
# TODO: Create a 'Revenue' column (Quantity * Price)
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']

# TODO: Create 'Month' and 'DayOfWeek' columns
df_clean['Month'] = df_clean['Date'].dt.month_name()
df_clean['DayOfWeek'] = df_clean['Date'].dt.day_name()

df_clean.head()

## 4. Aggregation and Analysis
Answer the following business questions using Pandas aggregations.

In [ ]:
# Question 1: What is the total revenue for the year?
total_revenue = df_clean['Revenue'].sum()
print(f"Total Revenue: ${total_revenue:,.2f}")

# Question 2: Which product generated the most revenue?
product_revenue = df_clean.groupby('ProductID')['Revenue'].sum().sort_values(ascending=False)
print("\nRevenue by Product:")
print(product_revenue)

## 5. Visualization
Create a bar chart showing revenue by product, and a line chart showing revenue over time (resampled by month).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart: Revenue by Product
sns.barplot(x=product_revenue.index, y=product_revenue.values, ax=axes[0], palette="viridis")
axes[0].set_title("Total Revenue by Product")
axes[0].set_ylabel("Revenue ($)")

# Line chart: Monthly Revenue
monthly_revenue = df_clean.set_index('Date').resample('M')['Revenue'].sum()
axes[1].plot(monthly_revenue.index.strftime('%b'), monthly_revenue.values, marker='o', color='b', linewidth=2)
axes[1].set_title("Monthly Revenue Trend")
axes[1].set_ylabel("Revenue ($)")

plt.tight_layout()
plt.show()